In [2]:
import numpy as np
import torch
import os
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

In [4]:
# Load the data from the specified directory
X_train = np.load('../ProcessedInputData/single_X_train.npy')
Y_train = np.load('../ProcessedInputData/single_y_train.npy')
X_val = np.load('../ProcessedInputData/single_X_val.npy')
Y_val = np.load('../ProcessedInputData/single_y_val.npy')


# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
Y_val_tensor = torch.tensor(Y_val, dtype=torch.long)

# Create a DataLoader for batching
batch_size = 64
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False) # don't shuffle for LTSM?
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

sanity_dataset = TensorDataset(X_train_tensor[:20], Y_train_tensor[:20]) # take just a few elements
sanity_loader = DataLoader(sanity_dataset, batch_size=5, shuffle=False)

In [ ]:
# Print the first few elements of the tensors
print("First few elements of Y_train_tensor:")
print(Y_train_tensor.size())
print(Y_train_tensor[:30])  # Print first 10 elements

print("First few elements of X_train_tensor:")
print(X_train_tensor.size())
# print(X_train_tensor[:3])  # Print first 3 elements

# each input is 5x90x1 matrix
# 5 bc there are 5 eeg channels recordings per sample,
# 90 bc i did fourier transform and binned it into 90 different bins from 0.5 to 40 Hz (relevant frequencies for EEG)
# the 1 was added just to be the dimension of picture style for CNN so its like black and white

# Update: now 90x1 where each channel is separate datapoint

print("Val tensors:")
print(Y_val_tensor.size())
print(X_val_tensor.size())

# original size [3153, 5, 90, 1]
# current size  [15768, 90, 1]

First few elements of Y_train_tensor:
torch.Size([15768])
tensor([1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0,
        1, 0, 1, 1, 0, 0])
First few elements of X_train_tensor:
torch.Size([15768, 90, 1])
Val tensors:
torch.Size([1971])
torch.Size([1971, 90, 1])


In [ ]:
# Conv2d expecs (batch size, channels, height, width) 4D input

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

def init_weights(m):
    if isinstance(m, nn.Linear) or isinstance(m, nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=(3, 1), padding=1)  # Adjusted kernel size
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(3, 1), padding=1)  # Adjusted kernel size
        self.pool = nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))  # Adjusted pooling size
        self.relu = nn.ReLU()

        # Compute correct FC input size dynamically
        self._to_linear = None
        self._compute_linear_input_size()

        self.fc1 = nn.Linear(self._to_linear, 64)
        self.fc2 = nn.Linear(64, 1)

    def _compute_linear_input_size(self):
        with torch.no_grad():
            x = torch.zeros(1, 1, 90, 1)  # Adjusted dummy input shape
            x = self.pool(self.relu(self.conv1(x)))
            x = self.pool(self.relu(self.conv2(x)))
            self._to_linear = x.numel()

    def forward(self, x):
        x = x.view(-1, 1, 90, 1)  # Adjusted reshaping
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x  # No sigmoid!

# Training Function
def train_model(model, train_loader, epochs=10):
    torch.manual_seed(1000)
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            labels = labels.view(-1, 1).float()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            predicted = torch.round(torch.sigmoid(outputs))
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        avg_loss = running_loss / len(train_loader)
        accuracy = 100 * correct / total
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

# Initialize Model
cnn_model = CNN()
cnn_model.apply(init_weights)

# Train Model
train_model(model=cnn_model, train_loader=sanity_loader, epochs=10)


Epoch [1/10], Loss: 1.0507, Accuracy: 65.00%
Epoch [2/10], Loss: 14.2156, Accuracy: 55.00%
Epoch [3/10], Loss: 5.0312, Accuracy: 45.00%
Epoch [4/10], Loss: 0.7551, Accuracy: 55.00%
Epoch [5/10], Loss: 0.6762, Accuracy: 55.00%
Epoch [6/10], Loss: 0.6527, Accuracy: 55.00%
Epoch [7/10], Loss: 0.7276, Accuracy: 60.00%
Epoch [8/10], Loss: 0.9865, Accuracy: 60.00%
Epoch [9/10], Loss: 0.6776, Accuracy: 65.00%
Epoch [10/10], Loss: 0.6698, Accuracy: 65.00%
